# 02 - EDA Business & Storytelling (Slide-ready)

Ce notebook repond a des questions business clefs pour JO 2028.
Chaque section contient:
- une question business
- un graphique exploitable en soutenance
- un angle narratif YPerf


## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

sns.set_theme(style="darkgrid")
pd.set_option("display.max_columns", 60)

DATA_PATH = Path("../data/raw/olympics_dataset.csv")
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

raw = pd.read_csv(DATA_PATH)
raw["Medal"] = raw["Medal"].fillna("None")
raw["is_medal"] = (raw["Medal"] != "None").astype(int)
raw["medal_points"] = raw["Medal"].map({"Gold": 3, "Silver": 2, "Bronze": 1}).fillna(0).astype(int)

raw.head(2)


## Q1. Quelle est la dynamique globale de la competition olympique?
**Narratif:** le niveau d'intensite (participations + medailles) evolue dans le temps, ce qui impacte la difficulte de prediction.


In [ ]:
year_kpis = (
    raw.groupby("Year", as_index=False)
    .agg(entries=("player_id", "count"), medals=("is_medal", "sum"), countries=("NOC", "nunique"))
    .sort_values("Year")
)

fig = px.line(year_kpis, x="Year", y=["entries", "medals"], markers=True,
              title="Global olympic activity over time")
fig.show()
year_kpis.tail(10)


## Q2. Quels pays dominent historiquement les medailles?
**Narratif:** identifier les acteurs structurellement forts (base line-up des favoris 2028).


In [ ]:
country_medals = (
    raw.groupby("NOC", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(country_medals.head(15), x="NOC", y="medals", title="Top 15 countries by total medals")
fig.show()
country_medals.head(15)


## Q3. Quels pays sont en acceleration recente (momentum)?
**Narratif:** differencier les leaders historiques des nations en progression depuis 2016.


In [ ]:
recent_start = 2016

country_year = (
    raw.groupby(["NOC", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

hist = country_year[country_year["Year"] < recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
recent = country_year[country_year["Year"] >= recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

momentum = hist.merge(recent, on="NOC", how="outer").fillna(0)
momentum["delta"] = momentum["recent_avg"] - momentum["hist_avg"]
momentum = momentum[momentum["recent_avg"] >= 5].sort_values("delta", ascending=False)

fig = px.bar(momentum.head(15), x="NOC", y="delta", title="Countries with strongest recent medal momentum")
fig.show()
momentum.head(15)


## Q4. Quels sports concentrent la creation de medailles?
**Narratif:** prioriser les sports a fort volume pour maximiser l'impact de prediction.


In [ ]:
sport_medals = (
    raw.groupby("Sport", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(sport_medals.head(15), x="Sport", y="medals", title="Top sports by medal volume")
fig.update_layout(xaxis_tickangle=-35)
fig.show()
sport_medals.head(15)


## Q5. Quels sports accelerent recemment?
**Narratif:** detecter les disciplines dont le poids recent augmente.


In [ ]:
sport_year = (
    raw.groupby(["Sport", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

sport_hist = sport_year[sport_year["Year"] < recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
sport_recent = sport_year[sport_year["Year"] >= recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

sport_momentum = sport_hist.merge(sport_recent, on="Sport", how="outer").fillna(0)
sport_momentum["delta"] = sport_momentum["recent_avg"] - sport_momentum["hist_avg"]
sport_momentum = sport_momentum.sort_values("delta", ascending=False)

fig = px.bar(sport_momentum.head(15), x="Sport", y="delta", title="Sports with strongest recent acceleration")
fig.update_layout(xaxis_tickangle=-35)
fig.show()
sport_momentum.head(15)


## Q6. Quelle est la repartition du niveau de performance (Gold/Silver/Bronze)?
**Narratif:** la structure des podiums informe la robustesse competitive.


In [ ]:
medal_mix = (
    raw[raw["Medal"] != "None"]
    .groupby("Medal", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

fig = px.pie(medal_mix, names="Medal", values="count", title="Global medal composition")
fig.show()
medal_mix


## Q7. Comment evolue la participation par genre?
**Narratif:** la convergence F/M est un signal de transformation structurelle.


In [ ]:
sex_year = (
    raw.groupby(["Year", "Sex"], as_index=False)
    .size()
    .rename(columns={"size": "entries"})
)

total_year = sex_year.groupby("Year", as_index=False)["entries"].sum().rename(columns={"entries": "total"})
sex_year = sex_year.merge(total_year, on="Year", how="left")
sex_year["share"] = sex_year["entries"] / sex_year["total"]

fig = px.line(sex_year, x="Year", y="share", color="Sex", markers=True,
              title="Gender participation share over time")
fig.show()
sex_year.tail(8)


## Q8. Quels pays performent le mieux en efficience (medailles / participations)?
**Narratif:** detecter les pays qui convertissent le mieux leurs engagements en podiums.


In [ ]:
country_eff = (
    raw.groupby("NOC", as_index=False)
    .agg(entries=("player_id", "count"), medals=("is_medal", "sum"))
)
country_eff["medal_rate"] = country_eff["medals"] / country_eff["entries"]
country_eff = country_eff[country_eff["entries"] >= 200].sort_values("medal_rate", ascending=False)

fig = px.bar(country_eff.head(15), x="NOC", y="medal_rate", title="Top countries by medal conversion rate (min 200 entries)")
fig.show()
country_eff.head(15)


## Q9. Quels pays sont specialises (dependance sport)?
**Narratif:** un portefeuille trop concentre augmente le risque en 2028.


In [ ]:
country_sport = (
    raw.groupby(["NOC", "Sport"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

country_total = country_sport.groupby("NOC", as_index=False)["medals"].sum().rename(columns={"medals": "total_medals"})
country_sport = country_sport.merge(country_total, on="NOC", how="left")
country_sport["share"] = country_sport["medals"] / country_sport["total_medals"]

specialization = (
    country_sport[country_sport["total_medals"] >= 50]
    .sort_values(["NOC", "share"], ascending=[True, False])
    .groupby("NOC", as_index=False)
    .first()[["NOC", "Sport", "share", "total_medals"]]
    .sort_values("share", ascending=False)
)

fig = px.bar(specialization.head(15), x="NOC", y="share", color="Sport",
             title="Countries with highest sport concentration (top sport share)")
fig.show()
specialization.head(15)


## Q10. Qui suivre pour 2028? (shortlist storytelling)
**Narratif:** combiner volume + momentum + efficience pour prioriser les nations a suivre.


In [ ]:
shortlist = (
    country_medals.rename(columns={"medals": "total_medals"})
    .merge(momentum[["NOC", "delta", "recent_avg"]], on="NOC", how="left")
    .merge(country_eff[["NOC", "medal_rate"]], on="NOC", how="left")
    .fillna({"delta": 0, "recent_avg": 0, "medal_rate": 0})
)

shortlist["score_2028"] = (
    0.45 * (shortlist["total_medals"] / shortlist["total_medals"].max()) +
    0.35 * (shortlist["delta"].clip(lower=0) / max(shortlist["delta"].clip(lower=0).max(), 1)) +
    0.20 * (shortlist["medal_rate"] / max(shortlist["medal_rate"].max(), 1e-9))
)

shortlist = shortlist.sort_values("score_2028", ascending=False)

fig = px.bar(shortlist.head(15), x="NOC", y="score_2028", title="YPerf shortlist score for JO 2028")
fig.show()

shortlist.head(15)


## Export des artefacts pour slides

Les tableaux sont exportes dans `reports/metrics/` pour insertion rapide dans le deck de soutenance.


In [ ]:
OUT = Path("../reports/metrics")
OUT.mkdir(parents=True, exist_ok=True)

country_medals.head(20).to_csv(OUT / "eda02_top_countries.csv", index=False)
momentum.head(20).to_csv(OUT / "eda02_country_momentum.csv", index=False)
sport_medals.head(20).to_csv(OUT / "eda02_top_sports.csv", index=False)
sport_momentum.head(20).to_csv(OUT / "eda02_sport_momentum.csv", index=False)
country_eff.head(20).to_csv(OUT / "eda02_country_efficiency.csv", index=False)
shortlist.head(20).to_csv(OUT / "eda02_shortlist_2028.csv", index=False)

print(f"Exports written to: {OUT.resolve()}")
